# Basic FastHTML Integration Example

This notebook demonstrates the minimal setup for using cjm-tqdm-capture with FastHTML and HTMX polling.

In [1]:
from fasthtml.common import *
import uuid, time

from cjm_tqdm_capture.progress_monitor import ProgressMonitor
from cjm_tqdm_capture.job_runner import JobRunner
from cjm_tqdm_capture.streaming import sse_stream

In [2]:
# For Jupyter display
from fasthtml.jupyter import JupyUvi, HTMX
from cjm_fasthtml_daisyui.core.testing import create_test_app, create_test_page, start_test_server
from cjm_fasthtml_daisyui.core.themes import DaisyUITheme
from IPython.display import display

# Import DaisyUI factories
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors, btn_behaviors
from cjm_fasthtml_daisyui.components.feedback.progress import progress, progress_colors

# Import Tailwind factories
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.typography import font_weight, font_size
from cjm_fasthtml_tailwind.utilities.sizing import w
from cjm_fasthtml_tailwind.core.base import combine_classes

In [3]:
# Create app
from cjm_fasthtml_daisyui.core.themes import DaisyUITheme
from cjm_fasthtml_daisyui.core.testing import create_test_app

app, rt = create_test_app(theme=DaisyUITheme.LIGHT)

# Initialize monitor and runner
monitor = ProgressMonitor()
runner = JobRunner(monitor)

In [4]:
# Define a simple task that uses tqdm
def simple_task():
    from tqdm import tqdm
    import time
    
    results = []
    for i in tqdm(range(100), desc="Processing"):
        time.sleep(0.01)
        results.append(i * 2)
    return results

In [5]:
# Create minimal UI with HTMX polling
from cjm_fasthtml_daisyui.core.testing import create_test_page

def render_start_button(disabled=False):
    """Render start button with appropriate state"""
    btn_classes = combine_classes(
        btn, 
        btn_colors.primary,
        btn_behaviors.disabled if disabled else ""
    )
    
    return Button(
        "Start Task", 
        id="start-btn",
        hx_post="/start_task" if not disabled else None,
        hx_target="#progress-container",
        hx_swap="innerHTML",
        cls=btn_classes,
        disabled=disabled,
        hx_swap_oob="true"  # Enable out-of-band swap
    )

@rt("/")
def index():
    return create_test_page(
        "Basic Progress Demo",
        Div(
            H2("Simple Progress Tracking"),
            render_start_button(disabled=False),
            Div(
                P("Ready", cls=combine_classes(font_weight.bold, m.t(4))),
                id="progress-container",
                cls=str(m.t(4))
            ),
            cls=str(p(8))
        )
    )

In [6]:
# API endpoints using HTMX polling with conditional updates
@rt("/start_task")
def start_task():
    job_id = str(uuid.uuid4())
    
    runner.start(
        job_id, 
        simple_task,
        patch_kwargs={
            "min_delta_pct": 5,
            "min_update_interval": 0.05,
            "emit_initial": True
        }
    )
    
    # Return initial UI with disabled button via out-of-band swap
    return Div(
        # Disable button via out-of-band swap
        render_start_button(disabled=True),
        # Progress display
        Div(
            P("Progress:", cls=combine_classes(font_weight.bold)),
            Progress(value="0", max="100", cls=combine_classes(progress, progress_colors.primary, w.full)),
            P("Starting...", cls=combine_classes(m.t(2), font_size.sm)),
            # Initial load only - polling added conditionally
            hx_get=f"/job_progress?job_id={job_id}",
            hx_trigger="load",
            hx_swap="innerHTML"
        )
    )

@rt("/job_progress")
def job_progress(job_id: str):
    """Returns current progress for a job"""
    snapshot = monitor.snapshot(job_id)
    
    if not snapshot:
        # Job not started yet - continue polling
        return Div(
            P("Progress:", cls=combine_classes(font_weight.bold)),
            Progress(value="0", max="100", cls=combine_classes(progress, progress_colors.primary, w.full)),
            P("Waiting...", cls=combine_classes(m.t(2), font_size.sm)),
            # Keep polling until task starts
            hx_get=f"/job_progress?job_id={job_id}",
            hx_trigger="load delay:100ms",
            hx_swap="outerHTML"
        )
    
    # Get first progress bar info
    bar_info = None
    if snapshot['bars']:
        first_bar = next(iter(snapshot['bars'].values()))
        bar_info = f"{first_bar.description}: {first_bar.progress:.1f}% ({first_bar.current}/{first_bar.total})"
    
    if snapshot['completed']:
        # Task finished - stop polling and re-enable button via out-of-band swap
        return Div(
            # Re-enable button via out-of-band swap
            render_start_button(disabled=False),
            # Final progress display
            P("Progress:", cls=combine_classes(font_weight.bold)),
            Progress(value="100", max="100", cls=combine_classes(progress, progress_colors.primary, w.full)),
            P("Completed!", cls=combine_classes(m.t(2), font_size.sm))
            # No hx_trigger - polling stops
        )
    
    # Task in progress - continue polling
    return Div(
        P("Progress:", cls=combine_classes(font_weight.bold)),
        Progress(
            value=str(snapshot['overall_progress']), 
            max="100", 
            cls=combine_classes(progress, progress_colors.primary, w.full)
        ),
        P(bar_info or f"Processing... {snapshot['overall_progress']:.1f}%", 
          cls=combine_classes(m.t(2), font_size.sm)),
        # Continue polling while in progress
        hx_get=f"/job_progress?job_id={job_id}",
        hx_trigger="load delay:100ms",
        hx_swap="outerHTML"
    )

# Optional: SSE endpoint still available if needed
@rt("/stream")
def stream(job_id: str):
    return EventStream(sse_stream(monitor, job_id, interval=0.1, heartbeat=10.0))

In [7]:
# Start server for Jupyter
from cjm_fasthtml_daisyui.core.testing import start_test_server
from fasthtml.jupyter import HTMX
from IPython.display import display

server = start_test_server(app)
display(HTMX())

In [8]:
# Stop server when done
server.stop()